In [1]:
pip install reportlab

Note: you may need to restart the kernel to use updated packages.


In [3]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, PageBreak, KeepTogether
)
from reportlab.platypus import ListFlowable, ListItem

PAGE_W, PAGE_H = A4
MARGIN = 18 * mm

# ── Color palette ──────────────────────────────────────────────
C_NAVY    = colors.HexColor("#1a2744")
C_BLUE    = colors.HexColor("#2563eb")
C_LTBLUE  = colors.HexColor("#dbeafe")
C_TEAL    = colors.HexColor("#0f766e")
C_LTTEAL  = colors.HexColor("#ccfbf1")
C_PURPLE  = colors.HexColor("#7c3aed")
C_LTPUR   = colors.HexColor("#ede9fe")
C_RED     = colors.HexColor("#dc2626")
C_LTRED   = colors.HexColor("#fee2e2")
C_AMBER   = colors.HexColor("#d97706")
C_LTAMB   = colors.HexColor("#fef3c7")
C_GREEN   = colors.HexColor("#16a34a")
C_LTGRN   = colors.HexColor("#dcfce7")
C_GRAY    = colors.HexColor("#f1f5f9")
C_DGRAY   = colors.HexColor("#64748b")
C_BLACK   = colors.HexColor("#0f172a")
C_WHITE   = colors.white

# ── Styles ──────────────────────────────────────────────────────
def make_styles():
    base = getSampleStyleSheet()
    s = {}

    s["cover_title"] = ParagraphStyle("cover_title",
        fontSize=28, leading=34, textColor=C_WHITE,
        fontName="Helvetica-Bold", alignment=TA_CENTER)

    s["cover_sub"] = ParagraphStyle("cover_sub",
        fontSize=13, leading=18, textColor=colors.HexColor("#bfdbfe"),
        fontName="Helvetica", alignment=TA_CENTER)

    s["ch_header"] = ParagraphStyle("ch_header",
        fontSize=18, leading=24, textColor=C_WHITE,
        fontName="Helvetica-Bold", alignment=TA_LEFT,
        spaceAfter=4)

    s["section"] = ParagraphStyle("section",
        fontSize=13, leading=18, textColor=C_NAVY,
        fontName="Helvetica-Bold", spaceBefore=10, spaceAfter=4,
        borderPad=2)

    s["subsection"] = ParagraphStyle("subsection",
        fontSize=11, leading=16, textColor=C_TEAL,
        fontName="Helvetica-Bold", spaceBefore=7, spaceAfter=3)

    s["body"] = ParagraphStyle("body",
        fontSize=9.5, leading=15, textColor=C_BLACK,
        fontName="Helvetica", spaceBefore=2, spaceAfter=4,
        alignment=TA_JUSTIFY)

    s["bullet"] = ParagraphStyle("bullet",
        fontSize=9.5, leading=14, textColor=C_BLACK,
        fontName="Helvetica", spaceBefore=1, spaceAfter=1,
        leftIndent=12, bulletIndent=2)

    s["formula"] = ParagraphStyle("formula",
        fontSize=9.5, leading=14, textColor=C_NAVY,
        fontName="Helvetica-Bold", alignment=TA_CENTER,
        spaceBefore=5, spaceAfter=5,
        backColor=C_GRAY, borderPad=6)

    s["note"] = ParagraphStyle("note",
        fontSize=9, leading=13, textColor=C_TEAL,
        fontName="Helvetica-Oblique", spaceBefore=3, spaceAfter=3)

    s["tip"] = ParagraphStyle("tip",
        fontSize=9.5, leading=14, textColor=colors.HexColor("#92400e"),
        fontName="Helvetica", spaceBefore=2, spaceAfter=2)

    s["table_hdr"] = ParagraphStyle("table_hdr",
        fontSize=9, leading=12, textColor=C_WHITE,
        fontName="Helvetica-Bold", alignment=TA_CENTER)

    s["table_cell"] = ParagraphStyle("table_cell",
        fontSize=8.5, leading=12, textColor=C_BLACK,
        fontName="Helvetica", alignment=TA_LEFT)

    s["toc_ch"] = ParagraphStyle("toc_ch",
        fontSize=11, leading=16, textColor=C_NAVY,
        fontName="Helvetica-Bold", spaceBefore=4)

    s["toc_sec"] = ParagraphStyle("toc_sec",
        fontSize=9.5, leading=14, textColor=C_DGRAY,
        fontName="Helvetica", leftIndent=12)

    s["page_num"] = ParagraphStyle("page_num",
        fontSize=8, textColor=C_DGRAY,
        fontName="Helvetica", alignment=TA_CENTER)

    s["keyword"] = ParagraphStyle("keyword",
        fontSize=9.5, leading=14, textColor=C_PURPLE,
        fontName="Helvetica-Bold")

    return s

S = make_styles()

# ── Helper flowables ─────────────────────────────────────────────
def HR(color=C_BLUE, t=0.5):
    return HRFlowable(width="100%", thickness=t, color=color, spaceAfter=4, spaceBefore=4)

def SP(h=6):
    return Spacer(1, h)

def B(text): return f"<b>{text}</b>"
def I(text): return f"<i>{text}</i>"
def C(text, color): return f'<font color="{color}">{text}</font>'

def chapter_header(num, title, color=C_NAVY):
    """Colored banner for chapter heading."""
    data = [[Paragraph(f"Chapter {num}", ParagraphStyle("ch_num",
                fontSize=10, textColor=colors.HexColor("#93c5fd"),
                fontName="Helvetica", leading=14)),
             ""],
            [Paragraph(title, S["ch_header"]), ""]]
    t = Table(data, colWidths=[PAGE_W - 2*MARGIN - 2, 2])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), color),
        ("ROWBACKGROUNDS", (0,0), (-1,-1), [color, color]),
        ("TOPPADDING",    (0,0), (-1,-1), 8),
        ("BOTTOMPADDING", (0,0), (-1,-1), 8),
        ("LEFTPADDING",   (0,0), (-1,-1), 12),
        ("RIGHTPADDING",  (0,0), (-1,-1), 8),
        ("SPAN", (0,0), (0,1)),
    ]))
    return [SP(10), t, SP(8)]

def info_box(text, bg=C_LTBLUE, border=C_BLUE, label=""):
    content = f"<b>{label}</b> {text}" if label else text
    data = [[Paragraph(content, ParagraphStyle("ib",
                fontSize=9.5, leading=14, textColor=C_BLACK,
                fontName="Helvetica", leftIndent=4))]]
    t = Table(data, colWidths=[PAGE_W - 2*MARGIN - 20])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), bg),
        ("BOX", (0,0), (-1,-1), 1.5, border),
        ("LEFTPADDING", (0,0), (-1,-1), 8),
        ("RIGHTPADDING", (0,0), (-1,-1), 8),
        ("TOPPADDING", (0,0), (-1,-1), 6),
        ("BOTTOMPADDING", (0,0), (-1,-1), 6),
    ]))
    return [t, SP(4)]

def tip_box(text):
    return info_box(text, bg=C_LTAMB, border=C_AMBER, label="💡 Exam Tip:")

def formula_box(text):
    return info_box(text, bg=C_GRAY, border=C_BLUE, label="📐 Formula:")

def warning_box(text):
    return info_box(text, bg=C_LTRED, border=C_RED, label="⚠️")

def two_col_table(headers, rows, col_widths=None):
    cw = col_widths or [(PAGE_W - 2*MARGIN)/2, (PAGE_W - 2*MARGIN)/2]
    data = [[Paragraph(h, S["table_hdr"]) for h in headers]]
    for row in rows:
        data.append([Paragraph(str(c), S["table_cell"]) for c in row])
    t = Table(data, colWidths=cw)
    t.setStyle(TableStyle([
        ("BACKGROUND",  (0,0), (-1,0), C_NAVY),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [C_WHITE, C_GRAY]),
        ("GRID",        (0,0), (-1,-1), 0.4, C_DGRAY),
        ("TOPPADDING",  (0,0), (-1,-1), 5),
        ("BOTTOMPADDING",(0,0),(-1,-1), 5),
        ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("RIGHTPADDING",(0,0), (-1,-1), 6),
        ("VALIGN",      (0,0), (-1,-1), "TOP"),
    ]))
    return [t, SP(6)]

def numeric_example(title, steps):
    """Highlighted worked-example block."""
    items = [Paragraph(f"<b>{title}</b>", ParagraphStyle("ne_title",
                fontSize=10, leading=14, textColor=C_TEAL,
                fontName="Helvetica-Bold"))]
    for s in steps:
        items.append(Paragraph(f"• {s}", ParagraphStyle("ne_step",
                fontSize=9, leading=13, textColor=C_BLACK,
                fontName="Helvetica", leftIndent=8)))
    data = [items]
    # flatten
    inner = [[i] for i in items]
    t = Table([[Paragraph("\n".join([
        f"<b>{title}</b>"] + [f"• {s}" for s in steps]),
        ParagraphStyle("ne_body", fontSize=9, leading=14, textColor=C_BLACK,
                fontName="Helvetica"))]],
        colWidths=[PAGE_W - 2*MARGIN - 20])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), C_LTTEAL),
        ("BOX", (0,0), (-1,-1), 1.2, C_TEAL),
        ("LEFTPADDING", (0,0), (-1,-1), 10),
        ("RIGHTPADDING",(0,0), (-1,-1), 10),
        ("TOPPADDING",  (0,0), (-1,-1), 8),
        ("BOTTOMPADDING",(0,0),(-1,-1), 8),
    ]))
    return [t, SP(5)]

def bullets(items, style=None):
    st = style or S["bullet"]
    return [Paragraph(f"• {i}", st) for i in items]

# ════════════════════════════════════════════════════════════════
#  CONTENT
# ════════════════════════════════════════════════════════════════

def cover_page():
    story = []
    # Big colored block
    data = [[
        Paragraph("MACHINE LEARNING APPLICATIONS", S["cover_title"]),
    ],[
        Paragraph("MAKAUT 6th Semester — Complete Study Guide", S["cover_sub"]),
    ],[
        Paragraph("Chapters: Regression · Classification · Ensemble · Model Evaluation",
                  S["cover_sub"]),
    ]]
    t = Table(data, colWidths=[PAGE_W - 2*MARGIN])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), C_NAVY),
        ("TOPPADDING",    (0,0), (-1,-1), 20),
        ("BOTTOMPADDING", (0,0), (-1,-1), 20),
        ("LEFTPADDING",   (0,0), (-1,-1), 24),
        ("RIGHTPADDING",  (0,0), (-1,-1), 24),
    ]))
    story += [SP(40), t, SP(20)]

    # Quick-reference badges
    badges = [
        ("Chapter 1", "Regression & Regularization", C_BLUE),
        ("Chapter 2", "Decision Trees & SVM", C_TEAL),
        ("Chapter 3", "Ensemble Learning", C_PURPLE),
        ("Chapter 4", "Model Evaluation", C_RED),
    ]
    badge_data = []
    for ch, title, col in badges:
        badge_data.append([
            Paragraph(f"<b>{ch}</b>", ParagraphStyle("bd_ch",
                fontSize=10, textColor=col, fontName="Helvetica-Bold",
                alignment=TA_CENTER)),
            Paragraph(title, ParagraphStyle("bd_ti",
                fontSize=9, textColor=C_BLACK, fontName="Helvetica",
                alignment=TA_CENTER)),
        ])

    bt = Table(badge_data, colWidths=[(PAGE_W-2*MARGIN)/2]*2)
    bt.setStyle(TableStyle([
        ("GRID",  (0,0), (-1,-1), 0.5, C_DGRAY),
        ("ROWBACKGROUNDS", (0,0), (-1,-1), [C_GRAY, C_WHITE]),
        ("TOPPADDING",    (0,0), (-1,-1), 8),
        ("BOTTOMPADDING", (0,0), (-1,-1), 8),
    ]))
    story += [bt, SP(16)]
    story.append(Paragraph("Prepared for MAKAUT Examinations | In-depth Theory + Numericals + Comparisons",
        ParagraphStyle("cover_footer", fontSize=9, textColor=C_DGRAY,
            fontName="Helvetica-Oblique", alignment=TA_CENTER)))
    story.append(PageBreak())
    return story


def chapter1():
    story = []
    story += chapter_header(1, "Supervised Learning – Regression & Regularization", C_BLUE)

    # ── 1.1 Linear Regression ──────────────────────────────────
    story.append(Paragraph("1.1 Linear Regression", S["section"]))
    story.append(Paragraph(
        "Linear Regression models the relationship between a dependent variable "
        "<b>y</b> and one or more independent variables <b>X</b> by fitting a "
        "straight line (or hyperplane) through the data.",
        S["body"]))

    story += formula_box("y = w<sub>0</sub> + w<sub>1</sub>x<sub>1</sub> + w<sub>2</sub>x<sub>2</sub> + ... + w<sub>n</sub>x<sub>n</sub>   or   y = X·w")

    story.append(Paragraph("Key Assumptions:", S["subsection"]))
    story += bullets([
        "Linearity – relationship between features and output is linear.",
        "Independence – observations are independent of each other.",
        "Homoscedasticity – constant variance of residuals.",
        "No multicollinearity – features are not highly correlated with each other.",
        "Normality of errors – residuals follow a Gaussian distribution.",
    ])

    story.append(Paragraph("Cost Function (Mean Squared Error):", S["subsection"]))
    story += formula_box("J(w) = (1/2m) * SUM [ (y_pred - y_actual)^2 ]   where m = number of samples")

    story.append(Paragraph(
        "The goal is to minimize J(w) by finding optimal weights <b>w</b>. "
        "This can be done analytically (closed-form) or iteratively (gradient descent).",
        S["body"]))

    story.append(Paragraph("1.1.1 Closed-Form Solution (Normal Equation)", S["subsection"]))
    story += formula_box("w = (X^T X)^(-1) X^T y")
    story.append(Paragraph(
        "Derivation: Set the gradient of J(w) w.r.t. w to zero. "
        "∂J/∂w = X^T(Xw - y) = 0  →  X^T Xw = X^T y  →  w = (X^T X)^(-1) X^T y. "
        "This is the Ordinary Least Squares (OLS) solution. "
        "Limitation: computing (X^T X)^(-1) is O(n^3) and fails if X^T X is singular.",
        S["body"]))

    # ── 1.2 Multivariate ───────────────────────────────────────
    story.append(Paragraph("1.2 Multivariate Linear Regression", S["section"]))
    story.append(Paragraph(
        "When there are multiple features (x_1, x_2, ..., x_n), the model "
        "extends naturally to a hyperplane in n-dimensional space. "
        "All formulas remain the same; X becomes an (m × n+1) design matrix "
        "with a column of ones appended for the bias/intercept term.",
        S["body"]))

    story += tip_box("For MAKAUT exams, you may be asked to write the matrix form y = Xw and derive w = (X^T X)^(-1) X^T y.")

    # ── 1.3 Logistic Regression ────────────────────────────────
    story.append(Paragraph("1.3 Logistic Regression", S["section"]))
    story.append(Paragraph(
        "Despite its name, Logistic Regression is a "
        "<b>classification</b> algorithm. It models the probability that an "
        "input belongs to a particular class using the sigmoid function.",
        S["body"]))

    story.append(Paragraph("The Sigmoid Function:", S["subsection"]))
    story += formula_box("σ(z) = 1 / (1 + e^(-z))   where  z = w^T x")
    story.append(Paragraph(
        "Output ranges between 0 and 1. If σ(z) ≥ 0.5 → predict class 1; else class 0. "
        "The decision boundary is the set of points where z = 0.",
        S["body"]))

    story.append(Paragraph("Loss Function (Binary Cross-Entropy / Log Loss):", S["subsection"]))
    story += formula_box("L = -[y log(p) + (1-y) log(1-p)]   where p = σ(z)")
    story.append(Paragraph(
        "Logistic Regression uses Maximum Likelihood Estimation (MLE) and is "
        "optimized via gradient descent (no closed-form solution).",
        S["body"]))

    story.append(Paragraph("Linear vs Logistic Regression — Comparison Table:", S["subsection"]))
    story += two_col_table(
        ["Feature", "Linear Regression", "Logistic Regression"],
        [
            ["Output", "Continuous value (e.g., price)", "Probability → Class (0 or 1)"],
            ["Task", "Regression", "Classification"],
            ["Function", "Identity / Linear", "Sigmoid"],
            ["Loss", "MSE / MAE", "Log Loss / Cross-Entropy"],
            ["Optimization", "Closed-form or GD", "Gradient Descent (MLE)"],
            ["Decision Boundary", "N/A", "Linear boundary in feature space"],
            ["Assumption", "Normally distributed errors", "Bernoulli distributed output"],
        ],
        col_widths=[90, 200, 200]
    )
    story += warning_box(
        "MAKAUT TRAP: Logistic Regression does NOT predict continuous values. "
        "It outputs a probability and then predicts a class. True/False questions often target this.")

    # ── 1.4 Regularization ─────────────────────────────────────
    story.append(Paragraph("1.4 Regularization", S["section"]))
    story.append(Paragraph(
        "Regularization adds a penalty term to the cost function to prevent "
        "overfitting by discouraging large weight values.",
        S["body"]))

    story.append(Paragraph("Why Regularize?", S["subsection"]))
    story += bullets([
        "High model complexity → learns noise → overfits training data.",
        "Regularization constrains the model, improving generalization.",
        "Introduces a bias-variance tradeoff (slight bias increase, large variance reduction).",
    ])

    # L1
    story.append(Paragraph("1.4.1 L1 Regularization (Lasso)", S["subsection"]))
    story += formula_box("J(w) = MSE + λ * SUM|w_i|   (L1 norm of weights)")
    story += bullets([
        "Adds the absolute values of weights as penalty.",
        "Produces SPARSE solutions — many weights become exactly 0.",
        "Acts as automatic feature selection (irrelevant features get zero weight).",
        "Not differentiable at w=0; requires subgradient methods.",
        "Preferred when many features are irrelevant.",
    ])

    # L2
    story.append(Paragraph("1.4.2 L2 Regularization (Ridge)", S["subsection"]))
    story += formula_box("J(w) = MSE + λ * SUM(w_i^2)   (L2 norm squared of weights)")
    story += bullets([
        "Adds squared values of weights as penalty.",
        "Weights are SHRUNK towards zero but rarely become exactly 0.",
        "Has a closed-form solution — computationally efficient.",
        "Performs well when most features are relevant.",
        "Also called 'Weight Decay' in neural networks.",
    ])

    # Comparison table
    story.append(Paragraph("L1 vs L2 Regularization — Quick Comparison:", S["subsection"]))
    story += two_col_table(
        ["Property", "L1 (Lasso)", "L2 (Ridge)"],
        [
            ["Penalty term", "λ * Σ|w_i|", "λ * Σw_i²"],
            ["Weight values", "Sparse (many = 0)", "Small but non-zero"],
            ["Feature selection", "Yes (built-in)", "No"],
            ["Solution", "No closed-form", "Closed-form exists"],
            ["Geometry", "Diamond constraint", "Circular constraint"],
            ["Best when", "Few relevant features", "Many relevant features"],
        ],
        col_widths=[120, 170, 170]
    )

    # Ridge derivation
    story.append(Paragraph("1.4.3 Closed-Form Solution for Ridge Regression (MLE Derivation)", S["subsection"]))
    story.append(Paragraph(
        "Starting from J(w) = (Xw - y)^T(Xw - y) + λw^T w, take derivative w.r.t. w and set to zero:",
        S["body"]))
    steps = [
        "∂J/∂w = 2X^T(Xw - y) + 2λw = 0",
        "X^T Xw - X^T y + λw = 0",
        "(X^T X + λI) w = X^T y",
        "w_ridge = (X^T X + λI)^(-1) X^T y",
        "Adding λI ensures (X^T X + λI) is always invertible — even when X^T X is singular!",
    ]
    story += numeric_example("Ridge Regression Derivation Steps", steps)

    story.append(Paragraph("Weight Decay Interpretation:", S["subsection"]))
    story.append(Paragraph(
        "In gradient descent, the Ridge update rule is: "
        "w ← w - η(2X^T(Xw-y) + 2λw) = w(1 - 2ηλ) - 2ηX^T(Xw-y). "
        "The factor (1 - 2ηλ) multiplies w at each step, effectively "
        "<b>decaying</b> the weights towards zero. Hence the name 'weight decay'.",
        S["body"]))

    story += tip_box(
        "Exam focus: Memorize w_ridge = (X^T X + λI)^(-1) X^T y. "
        "Know that L1 → sparsity, L2 → shrinkage. "
        "'Which regularization yields sparse weights?' → Answer: L1 (Lasso).")

    story.append(PageBreak())
    return story


def chapter2():
    story = []
    story += chapter_header(2, "Supervised Learning – Classification: Decision Trees & SVM", C_TEAL)

    # ── 2.1 Decision Trees ─────────────────────────────────────
    story.append(Paragraph("2.1 Decision Trees", S["section"]))
    story.append(Paragraph(
        "A Decision Tree is a flowchart-like tree structure where each "
        "internal node represents a test on a feature, each branch represents "
        "an outcome, and each leaf node represents a class label. "
        "It is a <b>supervised</b> algorithm used for both classification and regression.",
        S["body"]))

    story.append(Paragraph("Key Terminology:", S["subsection"]))
    story += bullets([
        "Root Node – the topmost node; represents the best splitting feature.",
        "Internal Node – a node that represents a test on a feature.",
        "Leaf Node – terminal node; holds the final class prediction.",
        "Stump – a decision tree with only one split (depth = 1); used in boosting.",
        "Depth – the length of the longest path from root to leaf.",
        "Pruning – technique to remove branches that add noise, reducing overfitting.",
    ])

    story.append(Paragraph("Properties:", S["subsection"]))
    story += bullets([
        "Can naturally handle non-linear decision boundaries.",
        "High variance — small changes in data can create completely different trees.",
        "Prone to overfitting without pruning.",
        "Easily interpretable (whitebox model).",
        "No need for feature scaling.",
    ])

    # Attribute selection
    story.append(Paragraph("2.1.1 Attribute Selection Measures", S["subsection"]))
    story.append(Paragraph(
        "Attribute selection determines which feature to split on at each node. "
        "The best split maximizes information gain or minimizes impurity.",
        S["body"]))

    # Entropy
    story.append(Paragraph("Entropy (used in ID3 algorithm):", S["subsection"]))
    story += formula_box("Entropy(S) = -SUM [ p_i * log2(p_i) ]   for each class i")
    story += bullets([
        "Entropy = 0 → perfectly pure (all samples same class).",
        "Entropy = 1 → maximally impure (equal split between classes for binary).",
        "Information Gain = Entropy(parent) - Weighted Average Entropy(children).",
    ])
    story += formula_box("IG(S, A) = Entropy(S) - SUM [ |S_v|/|S| * Entropy(S_v) ]")

    # Gini
    story.append(Paragraph("Gini Impurity (used in CART algorithm):", S["subsection"]))
    story += formula_box("Gini(S) = 1 - SUM [ p_i^2 ]   for each class i")
    story += bullets([
        "Gini = 0 → perfectly pure node.",
        "Gini = 0.5 → maximally impure (50-50 split for binary classification).",
        "Properties of Gini Impurity:",
        "  (a) Always between 0 and 0.5 for binary classification.",
        "  (b) Symmetric — does not penalize any particular class.",
        "  (c) Computationally faster than Entropy (no log computation).",
        "  (d) Tends to isolate the most frequent class in its own branch.",
    ])
    story += formula_box("Gini Gain = Gini(parent) - SUM [ (|S_v|/|S|) * Gini(S_v) ]")

    # Worked example
    story += numeric_example(
        "Worked Example — Calculating Gini Impurity",
        [
            "Dataset: 10 samples — 6 Class A, 4 Class B.",
            "p_A = 6/10 = 0.6,   p_B = 4/10 = 0.4",
            "Gini = 1 - (0.6² + 0.4²) = 1 - (0.36 + 0.16) = 1 - 0.52 = 0.48",
            "After split on feature X: Left node (4A, 1B), Right node (2A, 3B).",
            "Gini(Left)  = 1 - ((4/5)² + (1/5)²) = 1 - (0.64 + 0.04) = 0.32",
            "Gini(Right) = 1 - ((2/5)² + (3/5)²) = 1 - (0.16 + 0.36) = 0.48",
            "Weighted Gini = (5/10)*0.32 + (5/10)*0.48 = 0.16 + 0.24 = 0.40",
            "Gini Gain = 0.48 - 0.40 = 0.08  (positive → good split!)",
        ]
    )

    story += numeric_example(
        "Worked Example — Information Gain with Entropy",
        [
            "Dataset: 14 samples — 9 Yes, 5 No (Play Tennis).",
            "Entropy(S) = -(9/14)log2(9/14) - (5/14)log2(5/14)",
            "           = -(0.643 * -0.637) - (0.357 * -1.485)",
            "           = 0.410 + 0.530 = 0.940 bits",
            "If feature 'Wind' splits: Weak(9: 6Y 3N), Strong(5: 3Y 2N).",
            "Entropy(Weak)   = -(6/9)log2(6/9) - (3/9)log2(3/9) = 0.918",
            "Entropy(Strong) = -(3/5)log2(3/5) - (2/5)log2(2/5) = 0.971",
            "IG(Wind) = 0.940 - [(9/14)*0.918 + (5/14)*0.971] = 0.940 - 0.939 = 0.001",
        ]
    )

    # CART
    story.append(Paragraph("2.1.2 The CART Algorithm", S["subsection"]))
    story.append(Paragraph(
        "<b>CART</b> (Classification and Regression Trees) uses Gini Impurity "
        "for classification and MSE for regression. It always creates <b>binary splits</b>.",
        S["body"]))
    story += bullets([
        "Step 1: For each feature and each threshold, compute the weighted Gini after the split.",
        "Step 2: Choose the feature-threshold pair with the lowest weighted Gini (highest gain).",
        "Step 3: Recursively split child nodes until stopping criterion is met.",
        "Stopping criteria: max depth reached, min samples per leaf, no improvement in Gini.",
        "CART supports both classification (Gini) and regression (MSE) tasks.",
    ])

    # Overfitting & Pruning
    story.append(Paragraph("2.1.3 Overfitting and Pruning", S["subsection"]))
    story.append(Paragraph(
        "Deep trees memorize training data (high variance). Pruning reduces complexity:",
        S["body"]))
    story += bullets([
        "Pre-pruning (Early Stopping): Stop splitting if: max depth reached | samples < threshold | gain < min_gain.",
        "Post-pruning (Reduced Error Pruning): Grow full tree, then remove branches that do not improve validation accuracy.",
        "Cost-Complexity Pruning: Penalize tree by adding λ * |leaves| to cost.",
        "Decision trees can produce non-linear decision boundaries because each split creates axis-aligned partitions; combining many splits can approximate any boundary.",
    ])

    story += tip_box("MAKAUT often asks: 'Can decision trees produce non-linear boundaries?' → YES, because multiple splits create piecewise non-linear partitions.")

    # ── 2.2 SVM ────────────────────────────────────────────────
    story.append(Paragraph("2.2 Support Vector Machines (SVM)", S["section"]))
    story.append(Paragraph(
        "SVM finds the optimal hyperplane that maximally separates classes. "
        "It focuses only on the points closest to the decision boundary — "
        "the <b>support vectors</b>.",
        S["body"]))

    story.append(Paragraph("Core Concepts:", S["subsection"]))
    story += bullets([
        "Hyperplane: w^T x + b = 0  (the decision boundary).",
        "Positive margin: w^T x + b = +1  (support vectors of class +1).",
        "Negative margin: w^T x + b = -1  (support vectors of class -1).",
        "The Gutter / Margin = distance between the two margin hyperplanes.",
        "Support Vectors: data points that lie exactly on the margin boundaries.",
    ])

    story.append(Paragraph("Width of the Gutter (Margin):", S["subsection"]))
    story += formula_box("Width of Margin = 2 / ||w||   where ||w|| = sqrt(sum of w_i^2)")
    story.append(Paragraph(
        "Maximizing the margin = minimizing ||w||. "
        "This is a constrained optimization problem solved using Lagrange multipliers.",
        S["body"]))
    story += numeric_example(
        "Example: Finding Margin Width",
        [
            "Suppose w = [3, 4], then ||w|| = sqrt(9 + 16) = sqrt(25) = 5.",
            "Margin width = 2 / 5 = 0.4",
            "If w = [1, 0], then ||w|| = 1, Margin width = 2 / 1 = 2.",
            "Larger ||w|| → narrower margin. SVM minimizes ||w|| to maximize margin.",
        ]
    )

    story.append(Paragraph("Optimization Problem:", S["subsection"]))
    story += formula_box("Minimize: (1/2)||w||^2   Subject to: y_i(w^T x_i + b) >= 1  for all i")
    story += bullets([
        "This is a convex quadratic programming problem.",
        "Solved via Lagrange multipliers and KKT conditions.",
        "Only support vectors have non-zero Lagrange multipliers (α_i > 0).",
        "Final decision: f(x) = sign(SUM α_i y_i x_i^T x + b).",
    ])

    story.append(Paragraph("Soft-Margin SVM (C parameter):", S["subsection"]))
    story += formula_box("Minimize: (1/2)||w||^2 + C * SUM(ξ_i)   s.t. y_i(w^T x_i + b) >= 1 - ξ_i, ξ_i >= 0")
    story += bullets([
        "C controls the trade-off between margin width and misclassification.",
        "Large C → narrow margin, fewer errors (may overfit).",
        "Small C → wide margin, allows some misclassifications (better generalization).",
    ])

    # Kernel Trick
    story.append(Paragraph("2.2.1 The Kernel Trick", S["subsection"]))
    story.append(Paragraph(
        "When data is not linearly separable in the original feature space, "
        "SVM maps data to a higher-dimensional space using a kernel function, "
        "where it becomes linearly separable. "
        "The kernel trick computes dot products in the high-dim space "
        "<b>without explicitly computing the transformation</b>.",
        S["body"]))

    story += two_col_table(
        ["Kernel", "Formula", "Use Case"],
        [
            ["Linear", "K(x,z) = x^T z", "Linearly separable data"],
            ["Polynomial", "K(x,z) = (x^T z + c)^d", "Non-linear, moderate complexity"],
            ["RBF (Gaussian)", "K(x,z) = exp(-γ||x-z||²)", "Most common; handles complex boundaries"],
            ["Sigmoid", "K(x,z) = tanh(αx^T z + c)", "Similar to neural networks"],
        ],
        col_widths=[80, 180, 130]
    )

    # SVM vs LR
    story.append(Paragraph("2.2.2 SVM vs Logistic Regression — Detailed Comparison", S["subsection"]))
    story += two_col_table(
        ["Feature", "SVM", "Logistic Regression"],
        [
            ["Decision boundary", "Max-margin hyperplane", "Log-odds boundary"],
            ["Loss function", "Hinge loss", "Log loss (cross-entropy)"],
            ["Probability output", "No (class label only)", "Yes (probability via sigmoid)"],
            ["Outlier sensitivity", "Robust — boundary based on support vectors only", "Sensitive — all points influence boundary"],
            ["High-dim data", "Excellent (kernel trick)", "Good but may need regularization"],
            ["Large datasets", "Slow (O(n²) to O(n³))", "Fast, scales well"],
            ["Non-linear", "Yes (kernel trick)", "No (unless features engineered)"],
            ["Interpretability", "Moderate", "High (coefficient = feature weight)"],
            ["Regularization", "C parameter (inverse)", "L1/L2 regularization term"],
        ],
        col_widths=[110, 175, 175]
    )

    story.append(Paragraph("Why SVM is Fast and Accurate?", S["subsection"]))
    story += bullets([
        "Only support vectors determine the hyperplane — rest of the data is irrelevant after training.",
        "Kernel trick allows learning complex boundaries without expensive feature engineering.",
        "Max-margin principle provides theoretical guarantees on generalization.",
        "Convex optimization — guaranteed to find the global optimum.",
        "Effective in high-dimensional spaces (e.g., text classification).",
    ])

    story += tip_box(
        "MAKAUT Likely: Compare SVM and LR handling outliers → "
        "SVM is more robust because only support vectors define the boundary. "
        "Outliers far from the margin do not affect SVM unless they become support vectors.")

    story.append(PageBreak())
    return story


def chapter3():
    story = []
    story += chapter_header(3, "Ensemble Learning – Bagging & Boosting", C_PURPLE)

    story.append(Paragraph("3.1 Introduction to Ensemble Methods", S["section"]))
    story.append(Paragraph(
        "Ensemble methods combine multiple weak learners to create a strong learner. "
        "The key insight: many imperfect models together often outperform any single model.",
        S["body"]))

    # ── 3.2 Bagging ────────────────────────────────────────────
    story.append(Paragraph("3.2 Bagging (Bootstrap Aggregating)", S["section"]))
    story.append(Paragraph(
        "Bagging builds multiple models <b>independently and in parallel</b> "
        "on different random subsets of the training data (drawn with replacement — bootstrapping), "
        "then combines their predictions.",
        S["body"]))

    story += bullets([
        "Each model is trained on a bootstrap sample (~63% unique samples).",
        "For classification: final prediction = majority vote.",
        "For regression: final prediction = average of all predictions.",
        "Reduces variance significantly without increasing bias.",
        "Models are trained in parallel → efficient.",
        "Out-of-Bag (OOB) samples (~37%) serve as automatic validation set.",
    ])

    story.append(Paragraph("3.2.1 Random Forest (Bagging + Feature Randomness)", S["subsection"]))
    story.append(Paragraph(
        "Random Forest is the most popular bagging algorithm. "
        "In addition to bootstrapping samples, it also randomly selects a subset "
        "of features at each split, further de-correlating the trees.",
        S["body"]))
    story += bullets([
        "Number of features considered per split: sqrt(p) for classification, p/3 for regression.",
        "Very robust to overfitting.",
        "Provides feature importance scores.",
        "Handles missing data and high dimensionality well.",
    ])

    # ── 3.3 Boosting ───────────────────────────────────────────
    story.append(Paragraph("3.3 Boosting", S["section"]))
    story.append(Paragraph(
        "Boosting builds models <b>sequentially</b>. Each model corrects the "
        "errors of its predecessor. Misclassified samples get higher weights "
        "so the next model focuses more on them.",
        S["body"]))

    story += bullets([
        "Sequential training — each model depends on the previous one.",
        "Reduces both bias and variance.",
        "More prone to overfitting than bagging if not tuned carefully.",
        "Generally achieves higher accuracy than bagging.",
    ])

    # Bagging vs Boosting
    story.append(Paragraph("3.3.1 Bagging vs Boosting — Comparison Table", S["subsection"]))
    story += two_col_table(
        ["Property", "Bagging", "Boosting"],
        [
            ["Training order", "Parallel (independent)", "Sequential (dependent)"],
            ["Data sampling", "Bootstrap (with replacement)", "Weighted sampling"],
            ["Goal", "Reduce variance", "Reduce bias AND variance"],
            ["Models", "Same weight for all", "Later models get higher weight"],
            ["Overfitting", "Resistant", "Can overfit (use early stopping)"],
            ["Speed", "Faster (parallelizable)", "Slower (sequential)"],
            ["Example", "Random Forest", "AdaBoost, Gradient Boosting, XGBoost"],
        ],
        col_widths=[110, 175, 175]
    )
    story += info_box(
        "MAKAUT True/False: 'In boosting, trees are built sequentially.' → TRUE. "
        "In bagging, they are built in parallel. This distinction is a common exam target.",
        bg=C_LTPUR, border=C_PURPLE)

    # ── 3.4 AdaBoost ───────────────────────────────────────────
    story.append(Paragraph("3.4 AdaBoost (Adaptive Boosting) — Detailed Algorithm", S["section"]))
    story.append(Paragraph(
        "AdaBoost is the foundational boosting algorithm. It uses stumps "
        "(depth-1 trees) as weak classifiers and combines them with "
        "performance-weighted voting.",
        S["body"]))

    story.append(Paragraph("Algorithm Steps:", S["subsection"]))
    story += bullets([
        "Step 1: Initialize sample weights: w_i = 1/N  for all N samples.",
        "Step 2: For each round t = 1, 2, ..., T:",
        "   (a) Train weak learner h_t on weighted data.",
        "   (b) Compute weighted error: ε_t = SUM [ w_i * I(h_t(x_i) ≠ y_i) ] / SUM(w_i)",
        "   (c) Compute learner weight: α_t = 0.5 * ln((1 - ε_t) / ε_t)",
        "   (d) Update sample weights: w_i ← w_i * exp(-α_t * y_i * h_t(x_i))",
        "   (e) Normalize weights so they sum to 1.",
        "Step 3: Final prediction: H(x) = sign( SUM [ α_t * h_t(x) ] )",
    ])

    story += formula_box(
        "α_t = 0.5 * ln((1 - ε_t) / ε_t)   |   "
        "w_i(new) = w_i * exp(-α_t * y_i * h_t(x_i)) / Z_t"
    )

    # Worked numerical
    story.append(Paragraph("3.4.1 AdaBoost — Worked Numerical Example", S["subsection"]))
    story += numeric_example(
        "AdaBoost Numerical (10 Samples, y ∈ {+1, -1})",
        [
            "Step 1: Initialize weights: w_i = 1/10 = 0.1 for all 10 samples.",
            "Step 2 (Round 1): Train stump h_1. Suppose it misclassifies 3 samples.",
            "ε_1 = (0.1 + 0.1 + 0.1) = 0.3   (sum of weights of misclassified)",
            "α_1 = 0.5 * ln((1 - 0.3) / 0.3) = 0.5 * ln(2.333) = 0.5 * 0.847 = 0.424",
            "Update weights for CORRECT samples: w_i = 0.1 * exp(-0.424) = 0.1 * 0.655 = 0.0655",
            "Update weights for WRONG samples:   w_i = 0.1 * exp(+0.424) = 0.1 * 1.528 = 0.1528",
            "Normalize: divide each w_i by SUM of all new weights.",
            "Round 2: Train h_2 on the reweighted data (wrong samples now have more influence).",
            "Repeat for T rounds. Final: H(x) = sign(α_1*h_1(x) + α_2*h_2(x) + ...)",
        ]
    )

    story += tip_box(
        "MAKAUT often gives a table with data points, initial weights, and stump predictions. "
        "You need to: (1) compute ε, (2) compute α, (3) update and normalize weights. "
        "Practice this step-by-step calculation.")

    # ── 3.5 Gradient Boosting ──────────────────────────────────
    story.append(Paragraph("3.5 Gradient Boosting", S["section"]))
    story.append(Paragraph(
        "Gradient Boosting generalizes boosting to any differentiable loss function. "
        "Each new model is trained to predict the <b>residuals (pseudo-residuals)</b> "
        "of the current ensemble.",
        S["body"]))
    story += bullets([
        "Start with a constant prediction (e.g., mean of y).",
        "Compute residuals: r_i = y_i - F(x_i).",
        "Train a decision tree h_t to predict these residuals.",
        "Update: F(x) ← F(x) + η * h_t(x)  where η is the learning rate.",
        "Repeat for T iterations.",
        "XGBoost, LightGBM, CatBoost are highly optimized variants.",
    ])

    story.append(PageBreak())
    return story


def chapter4():
    story = []
    story += chapter_header(4, "Model Evaluation & Selection", C_RED)

    # ── 4.1 Train/Test Split ───────────────────────────────────
    story.append(Paragraph("4.1 Training Set vs. Test Set", S["section"]))
    story += bullets([
        "Training Set: Used to fit the model (learn parameters).",
        "Validation Set: Used to tune hyperparameters and select the model.",
        "Test Set: Held-out set; only used for final evaluation. NEVER train on this.",
        "Typical split: 70/15/15 or 80/20 (train/test).",
        "Cross-validation (k-fold): Rotate validation set k times for better estimates.",
    ])

    # ── 4.2 Confusion Matrix ───────────────────────────────────
    story.append(Paragraph("4.2 Confusion Matrix", S["section"]))
    story.append(Paragraph(
        "A confusion matrix summarizes the performance of a classifier. "
        "For binary classification (Positive/Negative):",
        S["body"]))

    cm_data = [
        [Paragraph("", S["table_hdr"]),
         Paragraph("Predicted Positive", S["table_hdr"]),
         Paragraph("Predicted Negative", S["table_hdr"])],
        [Paragraph("Actual Positive", S["table_cell"]),
         Paragraph("TP (True Positive)", ParagraphStyle("tp", fontSize=9, textColor=C_GREEN, fontName="Helvetica-Bold")),
         Paragraph("FN (False Negative)", ParagraphStyle("fn", fontSize=9, textColor=C_RED, fontName="Helvetica-Bold"))],
        [Paragraph("Actual Negative", S["table_cell"]),
         Paragraph("FP (False Positive)", ParagraphStyle("fp", fontSize=9, textColor=C_AMBER, fontName="Helvetica-Bold")),
         Paragraph("TN (True Negative)", ParagraphStyle("tn", fontSize=9, textColor=C_GREEN, fontName="Helvetica-Bold"))],
    ]
    cm = Table(cm_data, colWidths=[130, 170, 170])
    cm.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), C_NAVY),
        ("BACKGROUND", (0,1), (0,-1), C_GRAY),
        ("BACKGROUND", (1,1), (1,1), C_LTGRN),
        ("BACKGROUND", (2,2), (2,2), C_LTGRN),
        ("BACKGROUND", (1,2), (1,2), C_LTAMB),
        ("BACKGROUND", (2,1), (2,1), C_LTRED),
        ("GRID", (0,0), (-1,-1), 0.8, C_DGRAY),
        ("TOPPADDING", (0,0), (-1,-1), 8),
        ("BOTTOMPADDING", (0,0), (-1,-1), 8),
        ("LEFTPADDING", (0,0), (-1,-1), 8),
        ("ALIGN", (0,0), (-1,-1), "CENTER"),
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
    ]))
    story += [cm, SP(8)]

    story.append(Paragraph("Definitions:", S["subsection"]))
    story += bullets([
        "TP (True Positive): Predicted Positive, Actually Positive. ✓",
        "TN (True Negative): Predicted Negative, Actually Negative. ✓",
        "FP (False Positive) — Type I Error: Predicted Positive, Actually Negative. ✗",
        "FN (False Negative) — Type II Error: Predicted Negative, Actually Positive. ✗",
    ])

    # Metrics
    story.append(Paragraph("4.2.1 Performance Metrics — Formulas", S["subsection"]))
    story += formula_box("Accuracy  = (TP + TN) / (TP + TN + FP + FN)")
    story += formula_box("Precision = TP / (TP + FP)   [Of all predicted +ve, how many are truly +ve?]")
    story += formula_box("Recall (Sensitivity) = TP / (TP + FN)   [Of all actual +ve, how many detected?]")
    story += formula_box("Specificity = TN / (TN + FP)")
    story += formula_box("F1-Score = 2 * (Precision * Recall) / (Precision + Recall)   [Harmonic mean]")

    story += numeric_example(
        "Worked Example — Computing Metrics from Confusion Matrix",
        [
            "Given: TP=50, TN=30, FP=10, FN=5   (Total = 95 samples)",
            "Accuracy  = (50+30)/(50+30+10+5) = 80/95 = 0.842 = 84.2%",
            "Precision = 50/(50+10) = 50/60 = 0.833",
            "Recall    = 50/(50+5)  = 50/55 = 0.909",
            "F1-Score  = 2*(0.833*0.909)/(0.833+0.909) = 2*0.757/1.742 = 0.869",
            "When to use Precision: When FP is costly (e.g., spam filter).",
            "When to use Recall: When FN is costly (e.g., cancer detection).",
        ]
    )

    # ── 4.3 Overfitting & Underfitting ─────────────────────────
    story.append(Paragraph("4.3 Overfitting and Underfitting", S["section"]))
    story += two_col_table(
        ["", "Underfitting", "Overfitting"],
        [
            ["Also called", "High Bias", "High Variance"],
            ["Training error", "High", "Low"],
            ["Test error", "High", "High"],
            ["Model complexity", "Too simple", "Too complex"],
            ["Fit quality", "Misses patterns", "Memorizes noise"],
            ["Example", "Linear model on non-linear data", "Deep tree on small dataset"],
        ],
        col_widths=[100, 175, 175]
    )

    story += info_box(
        "MAKAUT True/False: 'If training error is low and validation error is also low, the model is good.' → TRUE. "
        "Low training + low validation error = well-fitted model.",
        bg=C_LTGRN, border=C_GREEN
    )

    story.append(Paragraph("How to Avoid Overfitting:", S["subsection"]))
    story += bullets([
        "1. Regularization (L1/L2) — penalize large weights.",
        "2. Reduce model complexity — fewer layers/depth.",
        "3. Get more training data — more diverse examples.",
        "4. Cross-validation — detect overfitting early.",
        "5. Dropout (neural networks) — randomly disable neurons during training.",
        "6. Early Stopping — monitor validation loss; stop when it starts increasing.",
        "7. Pruning (decision trees) — remove redundant branches.",
        "8. Data augmentation — artificially increase training data.",
        "9. Ensemble methods — bagging reduces variance significantly.",
    ])

    # ── 4.4 Bias-Variance Tradeoff ─────────────────────────────
    story.append(Paragraph("4.4 Bias-Variance Tradeoff", S["section"]))
    story += formula_box("Expected Error = Bias² + Variance + Irreducible Noise")
    story += bullets([
        "Bias: Error from wrong assumptions; simple models have high bias.",
        "Variance: Sensitivity to training data fluctuations; complex models have high variance.",
        "Irreducible Noise: Inherent noise in data; cannot be eliminated.",
        "As model complexity increases: Bias ↓, Variance ↑.",
        "Optimal model: Minimizes (Bias² + Variance).",
        "Bagging primarily reduces Variance.",
        "Boosting primarily reduces Bias (and also Variance).",
    ])

    story += tip_box(
        "The bias-variance tradeoff is the foundation of model selection. "
        "Simple models underfit (high bias); complex models overfit (high variance). "
        "Regularization and ensemble methods help navigate this tradeoff.")

    story.append(PageBreak())
    return story


def final_tips():
    story = []
    story += chapter_header(5, "Final Tips, Keywords & Quick Reference", C_NAVY)

    story.append(Paragraph("5.1 Key Keywords — Must Memorize", S["section"]))
    kw_data = [
        [Paragraph("Keyword", S["table_hdr"]), Paragraph("Definition", S["table_hdr"]), Paragraph("Chapter", S["table_hdr"])],
        ["Gutter", "The margin between the two support vector hyperplanes in SVM. Width = 2/||w||", "Ch. 2"],
        ["Kernel Trick", "Computing dot products in high-dim space without explicit transformation", "Ch. 2"],
        ["Pruning", "Removing branches from decision tree to reduce overfitting", "Ch. 2"],
        ["Stump", "Decision tree with only one split (depth=1). Used as weak learner in AdaBoost", "Ch. 2, 3"],
        ["Elbow Method", "Used in K-means to find optimal number of clusters (plot inertia vs k)", "Unsupervised"],
        ["One-Hot Encoding", "Converting categorical variables to binary vectors", "Preprocessing"],
        ["Weight Decay", "L2 regularization in neural networks; weights decay towards 0 each step", "Ch. 1"],
        ["Sparsity", "Most weights = 0; achieved by L1 (Lasso) regularization", "Ch. 1"],
        ["Support Vectors", "Data points closest to the decision boundary; define the hyperplane in SVM", "Ch. 2"],
        ["Bootstrap", "Sampling with replacement to create diverse training subsets", "Ch. 3"],
        ["OOB Error", "Out-of-Bag error — validation using samples not in bootstrap subset", "Ch. 3"],
    ]
    kw_tbl_data = [[Paragraph(str(r[0]), S["table_hdr"] if i==0 else S["table_cell"]),
                    Paragraph(str(r[1]), S["table_hdr"] if i==0 else S["table_cell"]),
                    Paragraph(str(r[2]), S["table_hdr"] if i==0 else S["table_cell"])]
                   for i, r in enumerate(kw_data)]
    kw_t = Table(kw_tbl_data, colWidths=[100, 300, 70])
    kw_t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), C_NAVY),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [C_WHITE, C_GRAY]),
        ("GRID", (0,0), (-1,-1), 0.4, C_DGRAY),
        ("TOPPADDING", (0,0), (-1,-1), 5),
        ("BOTTOMPADDING", (0,0), (-1,-1), 5),
        ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
    ]))
    story += [kw_t, SP(8)]

    story.append(Paragraph("5.2 Master Comparison Table", S["section"]))
    story += two_col_table(
        ["Topic", "A", "B", "Key Difference"],
        [
            ["Linear vs Logistic Reg.", "Continuous output", "Probability→Class", "Output type + loss function"],
            ["L1 vs L2", "Sparse weights (Lasso)", "Small weights (Ridge)", "L1→sparsity, L2→closed-form"],
            ["Bagging vs Boosting", "Parallel, reduces variance", "Sequential, reduces bias", "Order + error type reduced"],
            ["SVM vs LR", "Max-margin, kernel", "Probabilistic, fast", "Outlier robustness + prob."],
            ["Decision Tree vs SVM", "Interpretable, nonlinear", "Robust, kernel trick", "Overfitting vs margin"],
            ["Gini vs Entropy", "1 - Σp²  (fast)", "-Σp log(p)  (slower)", "Speed vs information purity"],
        ],
        col_widths=[105, 105, 105, 145]
    )

    story.append(Paragraph("5.3 True/False Quick-Fire (Common MAKAUT Questions)", S["section"]))
    tf_data = [
        ("Logistic Regression can predict continuous values.", "FALSE — it predicts class labels (0 or 1)."),
        ("In SVM, all training points contribute to the decision boundary.", "FALSE — only Support Vectors do."),
        ("Decision trees have high variance.", "TRUE — small data changes → very different trees."),
        ("L1 regularization yields sparse weights.", "TRUE — many weights become exactly 0."),
        ("Boosting trees are built sequentially.", "TRUE — each tree corrects previous errors."),
        ("Ridge Regression has a closed-form solution.", "TRUE — w = (X^T X + λI)^(-1) X^T y"),
        ("A stump is a tree with only one internal node.", "TRUE — it makes exactly one split."),
        ("Low training error AND low validation error = overfitting.", "FALSE — that's a good model. Overfitting = low train, HIGH validation error."),
        ("SVM uses Lagrange multipliers to find the optimal hyperplane.", "TRUE — KKT conditions via Lagrange."),
        ("Gini = 0 means a node is perfectly pure.", "TRUE — all samples belong to same class."),
    ]
    tf_table_data = [[Paragraph("Statement", S["table_hdr"]),
                      Paragraph("Answer", S["table_hdr"])]]
    for stmt, ans in tf_data:
        color = C_LTGRN if ans.startswith("TRUE") else C_LTRED
        tf_table_data.append([
            Paragraph(stmt, S["table_cell"]),
            Paragraph(ans, ParagraphStyle("tfa", fontSize=8.5, leading=12,
                textColor=C_BLACK, fontName="Helvetica")),
        ])
    tf_t = Table(tf_table_data, colWidths=[270, 200])
    tf_t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), C_NAVY),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [C_WHITE, C_GRAY]),
        ("GRID", (0,0), (-1,-1), 0.4, C_DGRAY),
        ("TOPPADDING", (0,0), (-1,-1), 5),
        ("BOTTOMPADDING", (0,0), (-1,-1), 5),
        ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
    ]))
    story += [tf_t, SP(8)]

    story.append(Paragraph("5.4 Derivations Cheat Sheet", S["section"]))
    story += formula_box("1. Normal Equation (OLS):    w = (X^T X)^(-1) X^T y")
    story += formula_box("2. Ridge Regression:          w = (X^T X + λI)^(-1) X^T y")
    story += formula_box("3. SVM Margin Width:          Margin = 2 / ||w||")
    story += formula_box("4. Entropy:                  H = -SUM p_i log2(p_i)")
    story += formula_box("5. Gini Impurity:            G = 1 - SUM p_i^2")
    story += formula_box("6. AdaBoost Learner Weight:  alpha = 0.5 * ln((1-epsilon)/epsilon)")
    story += formula_box("7. Sigmoid:                  sigma(z) = 1 / (1 + e^(-z))")
    story += formula_box("8. F1-Score:                 F1 = 2 * P * R / (P + R)")
    story += formula_box("9. Bias-Variance:            Error = Bias^2 + Variance + Noise")

    story.append(SP(10))
    story.append(HR(C_NAVY, 1.5))
    story.append(Paragraph(
        "MAKAUT Machine Learning Applications — Complete Study Guide | "
        "All chapters covered: Regression, Classification, Ensemble, Model Evaluation",
        ParagraphStyle("footer", fontSize=8, textColor=C_DGRAY,
            fontName="Helvetica-Oblique", alignment=TA_CENTER)))
    return story


# ════════════════════════════════════════════════════════════════
#  BUILD PDF
# ════════════════════════════════════════════════════════════════
def build_pdf(path):
    doc = SimpleDocTemplate(
        path,
        pagesize=A4,
        leftMargin=MARGIN, rightMargin=MARGIN,
        topMargin=MARGIN, bottomMargin=MARGIN,
        title="ML Applications — MAKAUT 6th Sem Study Guide",
        author="Study Guide Generator",
    )

    story = []
    story += cover_page()
    story += chapter1()
    story += chapter2()
    story += chapter3()
    story += chapter4()
    story += final_tips()

    doc.build(story)
    print(f"PDF created: {path}")


if __name__ == "__main__":
    build_pdf("ML_MAKAUT_StudyGuide.pdf")

PDF created: ML_MAKAUT_StudyGuide.pdf
